In [ ]:

import pandas as pd
import ipywidgets as widgets
from IPython.display import display

# Cau cie 10 

In [ ]:
import polars as pl
resumen_cie10=pl.read_csv("../../data/processed/resumen_egresos_v1.csv",infer_schema=0)

In [ ]:
import plotly.express as px
import pandas as pd
 
# 1. Convertimos a Pandas para la visualización
df_plot = resumen_cie10.to_pandas()

# Creamos una columna de fecha real para el eje X
df_plot["fecha_display"] = pd.to_datetime(
    df_plot["anio_egr"].astype(str) + "-" + df_plot["mes_egr"].astype(str) + "-01"
)

# 2. Crear la Visualización Dinámica
fig = px.line(
    df_plot, 
    x="fecha_display", 
    y="total_egresos", 
    color="entidad",      # Puedes cambiar esto por 'tipo' o 'clase'
    line_group="parr_ubi", 
    hover_data=["prov_ubi", "cant_ubi", "cau_cie10_std"],
    title="Evolución Temporal de Egresos Hospitalarios por Entidad",
    labels={
        "fecha_display": "Mes de Egreso",
        "total_egresos": "Número de Egresos",
        "entidad": "Entidad de Salud"
    },
    template="plotly_white"
)

# 3. Agregar filtros (Selectores de Diagnóstico y Provincia)
fig.update_layout(
    xaxis=dict(
        rangeselector=dict(
            buttons=list([
                dict(count=6, label="6m", step="month", stepmode="backward"),
                dict(count=1, label="1año", step="year", stepmode="backward"),
                dict(step="all")
            ])
        ),
        rangeslider=dict(visible=True), # Slider inferior para navegar el tiempo
        type="date"
    )
)

# Mostrar el gráfico
fig.show()

# CAU 221 rx mortalidad

In [ ]:
import polars as pl
resumen_cau_221rx=pl.read_csv("../../data/processed/resumen_egresos_v2.csv",infer_schema=0)

In [ ]:
import plotly.graph_objects as go

# 1. Preparar datos
df_plot = resumen_cau_221rx.to_pandas()
df_plot["fecha"] = pd.to_datetime(df_plot["anio_egr"].astype(str) + "-" + df_plot["mes_egr"].astype(str) + "-01")
df_plot['total_egresos'] = pd.to_numeric(df_plot['total_egresos'], errors='coerce')
# 2. Crear los Widgets de filtrado
# Filtro de Provincia
dropdown_prov = widgets.Dropdown(
    options=['TODAS'] + sorted(df_plot['prov_ubi'].unique().tolist()),
    value='TODAS', description='Provincia:'
)

# Filtro de Entidad (del grupo identificadores)
dropdown_entidad = widgets.Dropdown(
    options=['TODAS'] + sorted(df_plot['entidad'].unique().tolist()),
    value='TODAS', description='Entidad:'
)

# Filtro de Diagnóstico (Top 50 para no saturar el menú)
top_diag = df_plot["cau221rx_std"].unique().tolist()
dropdown_diag = widgets.Dropdown(
    options=['TODOS'] + df_plot["cau221rx_std"].unique().tolist(),
    value='TODOS', description='CAU-221RX:'
)


dropdown_mortalidad = widgets.Dropdown(
    options=['TODOS'] + sorted(df_plot['con_egrpa'].unique().tolist()),
    value='TODOS', description='Vivo o muerto:'
)

container = widgets.HBox([dropdown_prov, dropdown_entidad, dropdown_diag])
output = widgets.Output()

# 3. Función de actualización del gráfico
def update_chart(change):
    with output:
        output.clear_output(wait=True)
        
        # Filtrado dinámico
        dff = df_plot.copy()
        if dropdown_prov.value != 'TODAS':
            dff = dff[dff['prov_ubi'] == dropdown_prov.value]
        if dropdown_entidad.value != 'TODAS':
            dff = dff[dff['entidad'] == dropdown_entidad.value]
        if dropdown_diag.value != 'TODOS':
            dff = dff[dff['cau221rx_std'] == dropdown_diag.value]
        if dropdown_mortalidad.value !="TODOS":
            dff = dff[dff['con_egrpa'] == dropdown_diag.value]
            
        # Agrupar por fecha para la línea temporal
        resumen_temp = dff.groupby("fecha")["total_egresos"].sum().reset_index()
        
        fig = go.Figure()
        fig.add_trace(go.Scatter(
            x=resumen_temp["fecha"], 
            y=resumen_temp["total_egresos"],
            mode='lines+markers',
            line=dict(color='#2ca02c', width=3),
            marker=dict(size=8),
            name="Egresos"
        ))
        
        fig.update_layout(
            title=f"Tendencia: {dropdown_prov.value} | {dropdown_entidad.value} | {dropdown_diag.value}| {dropdown_mortalidad.value}",
            xaxis_title="Tiempo",
            yaxis_title="Total Egresos",
            template="plotly_white",
            height=500
        )
        fig.show()

# Vincular eventos
dropdown_prov.observe(update_chart, names='value')
dropdown_entidad.observe(update_chart, names='value')
dropdown_diag.observe(update_chart, names='value')
dropdown_mortalidad.observe(update_chart,names='values')

# Mostrar todo
display(container, output)
# Llamada inicial para mostrar el gráfico por primera vez
update_chart(None)

# CAU 221 rx mortalidad y edad

In [ ]:
import polars as pl
resumen_cau_221rx_edades=pl.read_csv("../../data/processed/resumen_egresos_v3.csv",infer_schema=0)
# Suponiendo que tu DataFrame es df
df = resumen_cau_221rx_edades.to_pandas()
df['edad_std'] = pd.to_numeric(df['edad_std'], errors='coerce')
df['total_egresos'] = pd.to_numeric(df['total_egresos'], errors='coerce')
df['anio_egr'] = pd.to_numeric(df['anio_egr'], errors='coerce')
df['cau221rx_std'] = df['cau221rx_std'].fillna('no asignado')

In [ ]:

# mapear las razones de con_egrpa
map={"Fallecido en 48 horas y más":"Fallecido",
     "Vivo":"Vivo",
     "Fallecido menos de 48 horas":"Fallecido"}
df["con_egrpa"] = df["con_egrpa"].map(map)

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import ipywidgets as widgets
from ipywidgets import interact
colores_con_egrpa = {
    'Fallecido': 'red',  # Cambia 'Valor1' por los valores de 'con_egrpa' y 'red' por el color que desees
    'Vivo': 'green',

    # Agrega más valores y colores según sea necesario
}


# Filtrar por cau221rx_std (esto lo haremos interactivo)
def plot_histogram(cau221rx_value,year_value):
    # Filtrar por el valor de cau221rx_std seleccionado
    df_filtered = df[df['cau221rx_std'] == cau221rx_value]  # Cambia el filtro según el valor del dropdown
    df_filtered = df_filtered[df_filtered['anio_egr']==year_value]
    # Crear la columna de rangos de edad
    bins = [1, 10, 20, 30, 40, 50, 60, 70, 80, 90, 100, 115]  # Definimos los rangos de edad
    labels = ['1-10', '11-20', '21-30', '31-40', '41-50', '51-60', '61-70', '71-80', '81-90', '91-100', '101-115']
    df_filtered['edad_rango'] = pd.cut(df_filtered['edad_std'], bins=bins, labels=labels, right=False)

    # Agrupar por año, rango de edad y con_egrpa, y calcular la suma de total_egresos
    df_grouped = df_filtered.groupby(['anio_egr', 'edad_rango', 'con_egrpa'])['total_egresos'].sum().reset_index()

    # Crear la visualización
    plt.figure(figsize=(14, 8))

    # Usamos seaborn para un gráfico de barras apiladas por 'con_egrpa'
    sns.histplot(data=df_grouped, x='edad_rango', hue='con_egrpa', weights='total_egresos', 
                 multiple="stack", discrete=True, kde=False, palette=colores_con_egrpa)

    # Ajustes de la visualización
    plt.title(f'Histograma de Egresos por Rango de Edad y Año (Filtrado por {cau221rx_value} y año {year_value})', fontsize=16)
    plt.xlabel('Rango de Edad', fontsize=12)
    plt.ylabel('Conteo de Egresos', fontsize=12)
    plt.xticks(rotation=45)
    plt.legend(title='Con EGRPA', title_fontsize='13', fontsize='11')

    # Mostrar el gráfico
    plt.tight_layout()
    #plt.show()

# Crear el dropdown interactivo para filtrar por 'cau221rx_std'
cau221rx_values = df['cau221rx_std'].unique()  # Obtener los valores únicos de 'cau221rx_std'
dropdown = widgets.Dropdown(
    options=cau221rx_values,
    description='Cau 221:',
    disabled=False
)

# Crear el dropdown interactivo para filtrar por 'cau221rx_std'
year_values = df['anio_egr'].unique()  # Obtener los valores únicos de 'cau221rx_std'
dropdown_year = widgets.Dropdown(
    options=year_values,
    description='Año:',
    disabled=False
)



In [ ]:

# Crear la interacción
interact(plot_histogram, cau221rx_value=dropdown,year_value=dropdown_year)


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import ipywidgets as widgets
from ipywidgets import interact

# Actualiza este diccionario con los nombres exactos que aparecen en tu print(df_grouped.head())
colores_con_egrpa = {
    'Fallecido': 'red',
    'Vivo': 'green'
}

def plot_histograms(cau221rx_value):
    # 1. Usar .copy() para evitar SettingWithCopyWarning
    df_filtered = df[df['cau221rx_std'] == cau221rx_value].copy()
    
    if df_filtered.empty:
        print("No hay datos para esta selección.")
        return

    # 2. Conversión segura
    df_filtered['edad_std'] = pd.to_numeric(df_filtered['edad_std'], errors='coerce')
    df_filtered['total_egresos'] = pd.to_numeric(df_filtered['total_egresos'], errors='coerce')
    
    bins = [1, 10, 20, 30, 40, 50, 60, 70, 80, 90, 100, 115]
    labels = ['1-10', '11-20', '21-30', '31-40', '41-50', '51-60', '61-70', '71-80', '81-90', '91-100', '101-115']
    df_filtered['edad_rango'] = pd.cut(df_filtered['edad_std'], bins=bins, labels=labels, right=False)

    # 3. Agrupación (observed=False para evitar el FutureWarning)
    df_grouped = df_filtered.groupby(['anio_egr', 'edad_rango', 'con_egrpa'], observed=False)['total_egresos'].sum().reset_index()

    # 4. Configurar FacetGrid
    g = sns.FacetGrid(data=df_grouped, col='anio_egr', col_wrap=5, height=4, sharex=True)
    
    # 5. CAMBIO CLAVE: Usar map_dataframe
    g.map_dataframe(
        sns.histplot, 
        x='edad_rango', 
        weights='total_egresos', 
        hue='con_egrpa', 
        multiple="stack", 
        discrete=True, 
        palette=colores_con_egrpa

    )

    # Ajustes finales
    g.set_axis_labels('Rango de Edad (años)', 'Conteo de Egresos')
    g.set_titles('Año {col_name}')
    g.add_legend(title="Estado del Paciente \n Verde:Vivo \n Rojo: Muerto")
    
    
    # Rotar etiquetas del eje X para que sean legibles
    for ax in g.axes.flat:
        ax.tick_params(axis='x', rotation=45)
        
    g.fig.suptitle(f'Egresos por Edad y Año - Filtro: {cau221rx_value}', fontsize=16)
    plt.subplots_adjust(top=0.85)
    plt.show()

# Dropdown e Interacción
df['cau221rx_std'].unique()
cau221rx_values = sorted(df['cau221rx_std'].unique())
dropdown = widgets.Dropdown(options=cau221rx_values, description='Cau 221:')


In [ ]:
#diabete obesidad cardiacos, hipertensivas
interact(plot_histograms, cau221rx_value=dropdown)

## Studio solo de bebes

In [ ]:
def calcular_edad_dias(row):
    cod = row['cod_edad']
    val = float(row['edad'])
    if cod.contains("Horas"): return val / 24
    if cod.contains("Días"):  return val
    if cod.contains("Meses"): return val * 30.44 # Promedio mensual
    if cod.contains("Años"):  return val * 365
    return val

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import ipywidgets as widgets
from ipywidgets import interact

# Actualiza este diccionario con los nombres exactos que aparecen en tu print(df_grouped.head())
colores_con_egrpa = {
    'Fallecido': 'red',
    'Vivo': 'green'
}

df_bebes=df[df['edad_std'] < 1].copy()


In [ ]:

def plot_histograms_bebes(cau221rx_value):
    # 1. Usar .copy() para evitar SettingWithCopyWarning
    df_filtered = df_bebes[df_bebes['cau221rx_std'] == cau221rx_value].copy()
    
    if df_filtered.empty:
        print("No hay datos para esta selección.")
        return

    # 2. Conversión segura
    df_filtered['edad_std'] = pd.to_numeric(df_filtered['edad_std'], errors='coerce')
    df_filtered['total_egresos'] = pd.to_numeric(df_filtered['total_egresos'], errors='coerce')
    
        # Definimos los límites en años
    # 7 días / 365 ≈ 0.019
    # 28 días / 365 ≈ 0.076
    # 6 meses / 12 ≈ 0.5
    bins = [0, 0.019, 0.076, 0.5, 1.0]

    labels = [
        'Neonato Precoz (0-6d)', 
        'Neonato Tardío (7-27d)', 
        'Lactante Menor (1-6m)', 
        'Lactante Mayor (6-12m)'
    ]
    df_filtered['edad_rango'] = pd.cut(df_filtered['edad_std'], bins=bins, labels=labels, right=False)

    # 3. Agrupación (observed=False para evitar el FutureWarning)
    df_grouped = df_filtered.groupby(['anio_egr', 'edad_rango', 'con_egrpa'], observed=False)['total_egresos'].sum().reset_index()

    # 4. Configurar FacetGrid
    g = sns.FacetGrid(data=df_grouped, col='anio_egr', col_wrap=5, height=4, sharex=True)
    
    # 5. CAMBIO CLAVE: Usar map_dataframe
    g.map_dataframe(
        sns.histplot, 
        x='edad_rango', 
        weights='total_egresos', 
        hue='con_egrpa', 
        multiple="stack", 
        discrete=True, 
        palette=colores_con_egrpa
    )

    # Ajustes finales
    g.set_axis_labels('Rango de Edad (años)', 'Conteo de Egresos')
    g.set_titles('Año {col_name}')
    g.add_legend() # Añade la leyenda para identificar los colores
    
    # Rotar etiquetas del eje X para que sean legibles
    for ax in g.axes.flat:
        ax.tick_params(axis='x', rotation=45)
        
    g.fig.suptitle(f'Egresos por Edad y Año - Filtro: {cau221rx_value}', fontsize=16)
    plt.subplots_adjust(top=0.85)
    plt.show()

# Dropdown e Interacción

top_50_bebes = df_bebes['cau221rx_std'].value_counts().nlargest(50).index.tolist()
dropdown_diag = widgets.Dropdown(
    options=['TODOS'] + sorted(top_50_bebes),
    value='TODOS', description='CAU 221 Bebés:'
)


In [ ]:
interact(plot_histograms_bebes, cau221rx_value=dropdown)